# SalesInsight-PY

## Criando o Dataset de vendas

In [2]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
 # Função abaixo foi usado para gerar um dataset randomico.
def gerar_dataset_vendas(n_registros=2000, seed=42):
    """Gera um dataset sintetico de vendas com dados sujos."""
    random.seed(seed)
    np.random.seed(seed)

    produtos = ["Notebook", "Smartphone", "Tablet", "Monitor",
                "Teclado", "Mouse", "Headset"]
    categorias = {"Notebook": "Computadores", "Smartphone": "Celulares",
                  "Tablet": "Celulares", "Monitor": "Computadores",
                  "Teclado": "Perifericos", "Mouse": "Perifericos",
                  "Headset": "Perifericos"}
    precos = {"Notebook": 3500, "Smartphone": 2200, "Tablet": 1800,
              "Monitor": 1200, "Teclado": 250, "Mouse": 120,
              "Headset": 350}
    regioes = ["Sudeste", "Sul", "Nordeste", "Centro-Oeste", "Norte"]

    data_inicio = datetime(2025, 1, 1)
    dados = []

    for i in range(n_registros):
        produto = random.choice(produtos)
        categoria = categorias[produto]
        quantidade = random.randint(1, 10)
        preco = round(precos[produto] * random.uniform(0.85, 1.15), 2)
        data = data_inicio + timedelta(days=random.randint(0, 364))
        data_txt = data.strftime("%Y-%m-%d")
        cliente = f"Cliente_{random.randint(1, 50):03d}"

        # --- sujeira proposital para a etapa de limpeza --- 
        if random.random() < 0.03:
        	data_txt = "DATA INVALIDA"       	# data invalida

        dados.append({
            "id_venda": i + 1,
            "data_venda": data_txt,
            "cliente": cliente,
            "produto": produto,
            "categoria": categoria,
            "regiao": random.choice(regioes),
            "quantidade": quantidade,
            "preco_unitario": preco,
        })

    return pd.DataFrame(dados)


# Gerar e salvar o CSV bruto
df_bruto = gerar_dataset_vendas()
df_bruto.to_csv("vendas.csv", index=False)
print(f"Dataset gerado com {len(df_bruto)} registros.")
print(df_bruto.head())

Dataset gerado com 2000 registros.
   id_venda     data_venda      cliente     produto     categoria    regiao  \
0         1     2025-05-21  Cliente_016       Mouse   Perifericos   Sudeste   
1         2  DATA INVALIDA  Cliente_003       Mouse   Perifericos       Sul   
2         3     2025-10-15  Cliente_013  Smartphone     Celulares     Norte   
3         4     2025-05-23  Cliente_001     Monitor  Computadores       Sul   
4         5     2025-03-21  Cliente_014       Mouse   Perifericos  Nordeste   

   quantidade  preco_unitario  
0           2          102.90  
1           9          105.13  
2           9         2267.33  
3           4         1181.72  
4           7          114.25  


## Inspecionar e Descrever os Dados

In [3]:
dados = pd.read_csv("vendas.csv") # ler o dataset gerado randomicamente
def inspecionar_dados(df):
    print("=== INSPECAO INICIAL DO DATASET ===")
    print("Shape:", df.shape) # verificar tamanho do indice e coluna do dataset
    print("Colunas:\n", list(df.columns)) # listar as colunas no dataset
    print("Tipos de dados:\n", df.dtypes) #quais os tipos de dados de cada coluna
    print("Valores nulos por coluna:\n", df.isnull().sum()) #verificação de valores nulos no dataset que no caso não ha nenhum registro vazio
    print("Primeiros registros:\n", df.head)
    return df

inspec_inicial = inspecionar_dados(dados) # chamando a função inspecionar dados da variavel dados ('vendas.csv') e declarando ela como inspeção inicial

=== INSPECAO INICIAL DO DATASET ===
Shape: (2000, 8)
Colunas:
 ['id_venda', 'data_venda', 'cliente', 'produto', 'categoria', 'regiao', 'quantidade', 'preco_unitario']
Tipos de dados:
 id_venda            int64
data_venda            str
cliente               str
produto               str
categoria             str
regiao                str
quantidade          int64
preco_unitario    float64
dtype: object
Valores nulos por coluna:
 id_venda          0
data_venda        0
cliente           0
produto           0
categoria         0
regiao            0
quantidade        0
preco_unitario    0
dtype: int64
Primeiros registros:
 <bound method NDFrame.head of       id_venda     data_venda      cliente     produto     categoria  \
0            1     2025-05-21  Cliente_016       Mouse   Perifericos   
1            2  DATA INVALIDA  Cliente_003       Mouse   Perifericos   
2            3     2025-10-15  Cliente_013  Smartphone     Celulares   
3            4     2025-05-23  Cliente_001     Monitor

## Tratar os Dados (datetime)

In [4]:
def tratar_dados(df):
     """  Trata o DataFrame de vendas.  Retorna: (df_limpo, relatorio), onde relatorio e um dicionario  
               com as contagens de registros iniciais, excluidos e finais.  """
     df = df.copy() # copiando o dataset para não alterar o arquivo original
     # Remover espacos extras nas colunas de texto
     df.columns = df.columns.str.strip()

     registros_iniciais = len(df) # atribuindo o tamanho do indice do dataset
     
     # Converter data_venda e descartar datas invalidas
     df["data_venda"] = pd.to_datetime(df["data_venda"], errors="coerce")
     registros_excluidos = df["data_venda"].isna().sum() # contabilizar os erros com .isna() e .sum() para contar os erros
     df_limpos = df.dropna(subset=["data_venda"]).copy() # entregando para a nova variavel df_limpos o dataset após excluir com o comando .dropna os erros encontrados
     
     # Ajustar os tipos numericos
     df_limpos["quantidade"] = df_limpos["quantidade"].astype(int)
     df_limpos["preco_unitario"] = df_limpos["preco_unitario"].astype(float)
     # print(df[["quantidade", "preco_unitario"]].dtypes)
     # Exibir os tipos numericos de quantidade e preco_unitario.
     
     # Montar e imprimir o relatorio de limpeza
     relatorio = {
          "registros_iniciais": registros_iniciais,
          "registros_excluidos": int(registros_excluidos),
          "registros_finais": len(df_limpos),
     }
     print(relatorio)

     return df_limpos, relatorio

dados_limpos, relatorio = tratar_dados(dados)

{'registros_iniciais': 2000, 'registros_excluidos': 65, 'registros_finais': 1935}


## Criar Colunas Derivadas com Transformações Condicionais

In [5]:
def criar_colunas_derivadas(df):
    """Cria as colunas derivadas a partir do DataFrame já limpo."""
    df = df.copy()

    # Receita total: quantidade x preço unitário
    df["receita_total"] = df["quantidade"] * df["preco_unitario"]

    # Mês numérico e nome do mês em português
    df["mes"] = df["data_venda"].dt.month
    meses_pt = {
        1: "Janeiro", 2: "Fevereiro", 3: "Março", 4: "Abril",
        5: "Maio", 6: "Junho", 7: "Julho", 8: "Agosto",
        9: "Setembro", 10: "Outubro", 11: "Novembro", 12: "Dezembro"
    }
    df["mes_nome"] = df["mes"].map(meses_pt)

    # Trimestre e ano
    df["trimestre"] = "Q" + df["data_venda"].dt.quarter.astype(str)
    df["ano"] = df["data_venda"].dt.year

    # Classificação condicional da receita por item
    df["faixa_receita_item"] = "Baixa"
    df.loc[df["receita_total"] >= 500, "faixa_receita_item"] = "Média"
    df.loc[df["receita_total"] >= 1000, "faixa_receita_item"] = "Alta"
    return df


# Aplicar as transformações somente após a limpeza das datas
dados_limpos = criar_colunas_derivadas(dados_limpos)

print("Colunas derivadas criadas com sucesso!")
print(dados_limpos[[
    "data_venda", "quantidade", "preco_unitario", "receita_total",
    "mes", "mes_nome", "trimestre", "ano", "faixa_receita_item"
]].head())

print("\nDistribuição por faixa de receita:")
print(dados_limpos["faixa_receita_item"].value_counts())

Colunas derivadas criadas com sucesso!
  data_venda  quantidade  preco_unitario  receita_total  mes mes_nome  \
0 2025-05-21           2          102.90         205.80    5     Maio   
2 2025-10-15           9         2267.33       20405.97   10  Outubro   
3 2025-05-23           4         1181.72        4726.88    5     Maio   
4 2025-03-21           7          114.25         799.75    3    Março   
5 2025-07-03           2         3373.92        6747.84    7    Julho   

  trimestre   ano faixa_receita_item  
0        Q2  2025              Baixa  
2        Q4  2025               Alta  
3        Q2  2025               Alta  
4        Q1  2025              Média  
5        Q3  2025               Alta  

Distribuição por faixa de receita:
faixa_receita_item
Alta     1532
Média     213
Baixa     190
Name: count, dtype: int64


## Calcular Métricas Agregadas com groupby

In [6]:
def calcular_metricas(df):
    """
    Calcula métricas agregadas do dataset.

    Retorna um dicionário de DataFrames com as chaves:
    por_mes, top_produtos, por_categoria e por_regiao.
    """

    metricas = {}

    # Receita total, quantidade vendida e número de vendas por mês
    metricas["por_mes"] = (df.groupby("mes").agg(receita_total=("receita_total", "sum"),
    quantidade=("quantidade", "sum"), n_vendas=("data_venda", "count")).sort_index().reset_index())

    # Top 5 produtos por receita total
    metricas["top_produtos"] = (df.groupby("produto").agg(receita_total=("receita_total", "sum"))
    .sort_values(by="receita_total",ascending=False).head(5).reset_index())

    # Receita total por categoria
    metricas["por_categoria"] = (df.groupby("categoria").agg(receita_total=("receita_total", "sum"))
    .sort_values(by="receita_total",ascending=False).reset_index())

    # Receita total e ticket médio por região
    metricas["por_regiao"] = (df.groupby("regiao").agg(receita_total=("receita_total", "sum"),
    ticket_medio=("receita_total", "mean")).sort_values(by="receita_total",ascending=False).reset_index())

    return metricas


# Calcular as métricas usando o dataset limpo
metricas = calcular_metricas(dados_limpos)

# Exibir os resultados no console
print("\n=== POR MÊS ===")
print(metricas["por_mes"].to_string(index=False))

print("\n=== TOP 5 PRODUTOS ===")
print(metricas["top_produtos"].to_string(index=False))

print("\n=== POR CATEGORIA ===")
print(metricas["por_categoria"].to_string(index=False))

print("\n=== POR REGIÃO ===")
print(metricas["por_regiao"].to_string(index=False))


=== POR MÊS ===
 mes  receita_total  quantidade  n_vendas
   1     1021509.43         809       156
   2     1245781.41         937       160
   3     1216221.11         867       157
   4     1076421.32         881       157
   5     1021307.32         773       140
   6     1006752.08         826       157
   7     1210764.88         827       158
   8     1241209.29         932       165
   9     1172255.68         893       168
  10     1337802.29        1042       186
  11     1124914.91         869       154
  12     1380269.87        1019       177

=== TOP 5 PRODUTOS ===
   produto  receita_total
  Notebook     4905935.08
Smartphone     3350290.74
    Tablet     2740096.52
   Monitor     1983396.40
   Headset      510623.14

=== POR CATEGORIA ===
   categoria  receita_total
Computadores     6889331.48
   Celulares     6090387.26
 Perifericos     1075490.85

=== POR REGIÃO ===
      regiao  receita_total  ticket_medio
         Sul     2917259.08   7738.087745
     Sudeste     2

## Segmentar Clientes por Nível de Gasto

In [7]:
def segmentar_clientes(df):
    """
    Agrupa por cliente, calcula o total gasto e classifica
    em Bronze, Prata ou Ouro.

    Retorna um DataFrame com:
    cliente, total_gasto, segmento.
    """

    # Agrupar clientes e calcular o gasto acumulado
    clientes = (df.groupby("cliente").agg(total_gasto=("receita_total", "sum")).reset_index())

    # Classificar clientes com lambda e apply
    clientes["segmento"] = clientes["total_gasto"].apply(lambda gasto:("Bronze" if gasto < 200000 else "Prata" if gasto <= 400000 else "Ouro"))

    # Ordenar do maior para o menor gasto
    clientes = clientes.sort_values(by="total_gasto", ascending=False).reset_index(drop=True)

    return clientes


# Executar a segmentação usando o dataset limpo
clientes_segmentados = segmentar_clientes(dados_limpos)

# Exibir os 10 maiores clientes
print("\n=== TOP 10 CLIENTES POR GASTO ===")
print(clientes_segmentados.head(10).to_string(index=False))

# Exibir a distribuição de clientes por segmento
print("\n=== DISTRIBUIÇÃO POR SEGMENTO ===")
print(clientes_segmentados["segmento"].value_counts().reindex(["Bronze", "Prata", "Ouro"], fill_value=0))

# Exibir a distribuição percentual
print("\n=== DISTRIBUIÇÃO PERCENTUAL ===")
print((clientes_segmentados["segmento"].value_counts(normalize=True).reindex(["Bronze", "Prata", "Ouro"], fill_value=0).mul(100).round(2)).astype(str) + "%")



=== TOP 10 CLIENTES POR GASTO ===
    cliente  total_gasto segmento
Cliente_043    490172.44     Ouro
Cliente_008    481333.70     Ouro
Cliente_049    427969.67     Ouro
Cliente_048    426967.67     Ouro
Cliente_037    423937.36     Ouro
Cliente_045    420368.06     Ouro
Cliente_003    397902.19    Prata
Cliente_025    379913.35    Prata
Cliente_020    368325.00    Prata
Cliente_014    361562.73    Prata

=== DISTRIBUIÇÃO POR SEGMENTO ===
segmento
Bronze    11
Prata     33
Ouro       6
Name: count, dtype: int64

=== DISTRIBUIÇÃO PERCENTUAL ===
segmento
Bronze    22.0%
Prata     66.0%
Ouro      12.0%
Name: proportion, dtype: str


## Criar Visualizações com Matplotlib

In [8]:

# %pip install matplotlib seaborn

import matplotlib.pyplot as plt
import seaborn as sns
import os

# Paleta de cores definida para o projeto
CORES = {
    "azul": "#264653",
    "verde": "#2A9D8F",
    "laranja": "#E76F51",
    "amarelo": "#E9C46A",
    "roxo": "#7B2CBF",
    "cinza": "#6C757D"
}

PALETA_CATEGORIAS = [
    "#264653", "#E76F51", "#2A9D8F",
    "#7B2CBF", "#E9C46A", "#457B9D",
    "#D62828", "#606C38"
]

# Configuração visual global
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["font.family"] = "DejaVu Sans"



def gerar_graficos(df, metricas, pasta_saida="outputs/graficos"):
    """Gera e salva os quatro gráficos exigidos pelo projeto em PNG."""
    os.makedirs(pasta_saida, exist_ok=True)

    # Dataset utilizado nas visualizações
    df_graficos = df.copy()

    # Agregação mensal ao longo de todo o período. Ano e mês juntos evitam misturar, por exemplo,janeiro de 2025 com janeiro de 2026.
    df_graficos["periodo"] = (df_graficos["data_venda"].dt.to_period("M"))

    por_mes_grafico = (df_graficos.groupby("periodo").agg(receita_total=("receita_total", "sum")).reset_index())
    por_mes_grafico["periodo"] = (por_mes_grafico["periodo"].astype(str))

    # Dados para os gráficos de produtos e regiões
    top_produtos = metricas["top_produtos"].copy()
    por_regiao = metricas["por_regiao"].copy()

    # Figura - receita total por mes
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(
        por_mes_grafico["periodo"],
        por_mes_grafico["receita_total"],
        color=CORES["verde"],
        marker="o",
        linewidth=2.5,
        markersize=6,
        label="Receita total"
    )

    ax.set_title("Receita Total por Mês", fontweight="bold")
    ax.set_xlabel("Mês/Ano")
    ax.set_ylabel("Receita Total (R$)")
    ax.legend()
    ax.grid(True, linestyle="--", alpha=0.35)
    ax.tick_params(axis="x", rotation=45)

    fig.tight_layout()
    fig.savefig(os.path.join(pasta_saida, "receita_por_mes.png"),dpi=150,bbox_inches="tight")
    plt.close(fig)

    # Figura - top 5 produtos
    fig, ax = plt.subplots(figsize=(12, 6))

    produtos_plot = top_produtos.sort_values("receita_total", ascending=True)

    ax.barh(
        produtos_plot["produto"].astype(str),
        produtos_plot["receita_total"],
        color=CORES["azul"],
        edgecolor="white",
        label="Receita por produto"
    )

    ax.set_title("Top 5 Produtos por Receita", fontweight="bold")
    ax.set_xlabel("Receita Total (R$)")
    ax.set_ylabel("Produto")
    ax.legend()
    ax.grid(axis="x", linestyle="--", alpha=0.35)

    fig.tight_layout()
    fig.savefig(os.path.join(pasta_saida, "top_produtos.png"),dpi=150,bbox_inches="tight")
    plt.close(fig)

    # Figura - quantidade e receita
    fig, ax = plt.subplots(figsize=(12, 6))

    categorias = (df_graficos["categoria"].fillna("Sem categoria").unique())

    cores_categoria = {categoria: PALETA_CATEGORIAS[i % len(PALETA_CATEGORIAS)]for i, categoria in enumerate(categorias)}

    for categoria in categorias:
        dados_categoria = df_graficos[df_graficos["categoria"].fillna("Sem categoria")== categoria]

        ax.scatter(
            dados_categoria["quantidade"],
            dados_categoria["receita_total"],
            color=cores_categoria[categoria],
            label=str(categoria),
            alpha=0.7,
            s=45,
            edgecolors="white",
            linewidths=0.4
        )

    ax.set_title("Quantidade Vendida vs. Receita por Item",fontweight="bold")
    ax.set_xlabel("Quantidade Vendida")
    ax.set_ylabel("Receita Total (R$)")
    ax.legend(title="Categoria", bbox_to_anchor=(1.02, 1), loc="upper left")
    ax.grid(True, linestyle="--", alpha=0.3)

    fig.tight_layout()
    fig.savefig(
        os.path.join(pasta_saida, "quantidade_vs_receita.png"),
        dpi=150,
        bbox_inches="tight"
    )
    plt.close(fig)

    # Figura painel de resumo 2x2
    fig, axes = plt.subplots(2, 2, figsize=(16, 11))

    # Receita mensal
    axes[0, 0].plot(
        por_mes_grafico["periodo"],
        por_mes_grafico["receita_total"],
        color=CORES["verde"],
        marker="o",
        linewidth=2,
        label="Receita total"
    )
    axes[0, 0].set_title("Receita por Mês")
    axes[0, 0].set_xlabel("Mês/Ano")
    axes[0, 0].set_ylabel("Receita (R$)")
    axes[0, 0].tick_params(axis="x", rotation=45)
    axes[0, 0].legend()
    axes[0, 0].grid(True, linestyle="--", alpha=0.3)

    # Top 5 produtos
    axes[0, 1].barh(
        produtos_plot["produto"].astype(str),
        produtos_plot["receita_total"],
        color=CORES["azul"],
        label="Receita por produto"
    )
    axes[0, 1].set_title("Top 5 Produtos")
    axes[0, 1].set_xlabel("Receita (R$)")
    axes[0, 1].set_ylabel("Produto")
    axes[0, 1].legend()
    axes[0, 1].grid(axis="x", linestyle="--", alpha=0.3)

    # Dispersão por categoria
    for categoria in categorias:
        dados_categoria = df_graficos[df_graficos["categoria"].fillna("Sem categoria")== categoria]

        axes[1, 0].scatter(
            dados_categoria["quantidade"],
            dados_categoria["receita_total"],
            color=cores_categoria[categoria],
            label=str(categoria),
            alpha=0.7,
            s=35
        )

    axes[1, 0].set_title("Quantidade vs. Receita")
    axes[1, 0].set_xlabel("Quantidade")
    axes[1, 0].set_ylabel("Receita (R$)")
    axes[1, 0].legend(title="Categoria", fontsize=8)
    axes[1, 0].grid(True, linestyle="--", alpha=0.3)

    # Receita por região
    regioes_plot = por_regiao.sort_values("receita_total", ascending=True)

    axes[1, 1].barh(
        regioes_plot["regiao"].astype(str),
        regioes_plot["receita_total"],
        color=CORES["laranja"],
        label="Receita por região"
    )
    axes[1, 1].set_title("Receita Total por Região")
    axes[1, 1].set_xlabel("Receita (R$)")
    axes[1, 1].set_ylabel("Região")
    axes[1, 1].legend()
    axes[1, 1].grid(axis="x", linestyle="--", alpha=0.3)

    # Título geral do painel
    fig.suptitle(
        "SalesInsight PY - Painel Resumo",
        fontsize=17,
        fontweight="bold"
    )

    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(
        os.path.join(pasta_saida, "painel_resumo.png"),
        dpi=150,
        bbox_inches="tight"
    )
    plt.close(fig)


    print(f"Gráficos salvos em: {os.path.abspath(pasta_saida)}")


## Organizar o Código em Funções Reutilizáveis

In [12]:

def processar_coluna(df, coluna, funcao_transformacao, nome_saida=None):
    """
    Aplica uma função de transformação a uma coluna do DataFrame.

    Parâmetros:
        df (DataFrame): conjunto de dados.
        coluna (str): nome da coluna que será transformada.
        funcao_transformacao (function): função aplicada a cada valor.
        nome_saida (str, opcional): nome da nova coluna.

    Retorna:
        DataFrame: conjunto de dados com a nova coluna.
    """
    if coluna not in df.columns:
        raise ValueError(f"A coluna '{coluna}' não existe no DataFrame.")

    nome_saida = nome_saida or f"{coluna}_transformado"

    df = df.copy()
    df[nome_saida] = df[coluna].apply(funcao_transformacao)

    return df


# converter a receita total para milhares de reais
dados_limpos = processar_coluna(dados_limpos,"receita_total",lambda x: round(x / 1000, 2),nome_saida="receita_em_milhares")

# classificar a quantidade vendida
dados_limpos = processar_coluna(dados_limpos,"quantidade",lambda q: "Alto Volume" if q > 5 else "Baixo Volume",nome_saida="perfil_volume")

# Conferir os resultados
print("=== DEMONSTRAÇÃO DE FUNÇÃO DE ORDEM SUPERIOR ===")

print(
    dados_limpos[
        [
            "receita_total",
            "receita_em_milhares",
            "quantidade",
            "perfil_volume"
        ]
    ].head(10).to_string(index=False)
)
dados_limpos.to_csv("vendas_final.csv", index=False)

=== DEMONSTRAÇÃO DE FUNÇÃO DE ORDEM SUPERIOR ===
 receita_total  receita_em_milhares  quantidade perfil_volume
        205.80                 0.21           2  Baixo Volume
      20405.97                20.41           9   Alto Volume
       4726.88                 4.73           4  Baixo Volume
        799.75                 0.80           7   Alto Volume
       6747.84                 6.75           2  Baixo Volume
        970.40                 0.97           8   Alto Volume
       2148.72                 2.15           6   Alto Volume
       2004.75                 2.00           5  Baixo Volume
        647.16                 0.65           6   Alto Volume
        757.68                 0.76           3  Baixo Volume


## Exportar Resultados em CSV e JSON

In [ ]:
import json
import os

def exportar_resultados(metricas, clientes, estatisticas, pasta_saida="outputs"):
    """Exporta cada tabela de métricas e a segmentação em CSV, além do JSON geral."""
    os.makedirs(pasta_saida, exist_ok=True)

    # Exportar cada DataFrame de métricas para um CSV separado
    for nome, tabela in metricas.items():
        caminho_csv = os.path.join(pasta_saida, f"{nome}.csv")
        tabela.to_csv(caminho_csv, index=False, encoding="utf-8-sig")

    # Exportar a segmentação de todos os clientes
    clientes.to_csv(
        os.path.join(pasta_saida, "segmentacao_clientes.csv"),
        index=False,
        encoding="utf-8-sig"
    )

    # Converter os valores numéricos para tipos compatíveis com JSON
    serializavel = {}
    for chave, valor in estatisticas.items():
        if isinstance(valor, (int, float)):
            serializavel[chave] = round(float(valor), 2)
        else:
            serializavel[chave] = valor

    caminho_json = os.path.join(pasta_saida, "estatisticas_gerais.json")
    with open(caminho_json, "w", encoding="utf-8") as arquivo:
        json.dump(serializavel, arquivo, indent=4, ensure_ascii=False)

    # Ler o JSON novamente para validar a exportação
    with open(caminho_json, "r", encoding="utf-8") as arquivo:
        conferencia = json.load(arquivo)

    print(f"CSV e JSON exportados para: {os.path.abspath(pasta_saida)}")
    print("\n=== ESTATÍSTICAS LIDAS DO JSON ===")
    print(json.dumps(conferencia, indent=4, ensure_ascii=False))
    return conferencia


estatisticas = {
    "total_registros": int(len(dados_limpos)),
    "total_clientes": int(dados_limpos["cliente"].nunique()),
    "receita_total": float(dados_limpos["receita_total"].sum()),
    "quantidade_total_vendida": int(dados_limpos["quantidade"].sum()),
    "preco_medio_unitario": float(dados_limpos["preco_unitario"].mean())
}

conferencia = exportar_resultados(
    metricas=metricas,
    clientes=clientes_segmentados,
    estatisticas=estatisticas
)



CSV e JSON exportados para: c:\Users\Douglas Jonatan\Documents\GitHub\salesinsight-py\outputs

=== ESTATÍSTICAS LIDAS DO JSON ===
{
    "total_registros": 1935.0,
    "total_clientes": 50.0,
    "receita_total": 14055209.59,
    "quantidade_total_vendida": 10675.0,
    "preco_medio_unitario": 1316.28
}


## Executar o Fluxo Completo (Ponto de Entrada)

In [11]:
def main():
    """Executa o fluxo completo do SalesInsight PY, na ordem correta."""
    print("=" * 60)
    print(" SALESINSIGHT PY - Análise de Dados de Vendas")
    print("=" * 60)

    arquivo_dados = "vendas.csv"
    if not os.path.exists(arquivo_dados):
        raise FileNotFoundError(
            "O arquivo vendas.csv não foi encontrado. Execute primeiro a célula "
            "que gera o dataset ou coloque o arquivo na pasta do notebook."
        )

    # Carregar e inspecionar os dados brutos
    print("\n[1/8] Carregando e inspecionando o dataset...")
    df_bruto = pd.read_csv(arquivo_dados)
    print(f"Registros originais: {len(df_bruto)}")
    print(df_bruto.head().to_string(index=False))
    print("\nTipos de dados:")
    print(df_bruto.dtypes)

    # Limpar datas e tipos antes de transformar, agregar ou plotar
    print("\n[2/8] Limpando e validando datas...")
    df_limpo, relatorio_limpeza = tratar_dados(df_bruto)
    print("Relatório de limpeza:", relatorio_limpeza)

    # Criar colunas derivadas
    print("\n[3/8] Criando colunas derivadas...")
    df_limpo = criar_colunas_derivadas(df_limpo)

    # Calcular métricas agregadas
    print("\n[4/8] Calculando métricas...")
    metricas = calcular_metricas(df_limpo)
    for nome, resultado in metricas.items():
        print(f"\n=== {nome.upper()} ===")
        print(resultado.to_string(index=False))

    # Segmentar todos os clientes
    print("\n[5/8] Segmentando todos os clientes...")
    clientes = segmentar_clientes(df_limpo)
    print(f"Total de clientes únicos: {len(clientes)}")
    print("\n=== TOP 10 CLIENTES POR GASTO ===")
    print(clientes.head(10).to_string(index=False))
    distribuicao = clientes["segmento"].value_counts().reindex(
        ["Bronze", "Prata", "Ouro"], fill_value=0
    )
    print("\n=== DISTRIBUIÇÃO POR SEGMENTO ===")
    print(distribuicao)
    print("\n=== DISTRIBUIÇÃO PERCENTUAL ===")
    print((distribuicao / max(len(clientes), 1) * 100).round(2).astype(str) + "%")

    # Aplicar transformações reutilizáveis
    print("\n[6/8] Aplicando transformações reutilizáveis...")
    df_limpo = processar_coluna(
        df_limpo, "receita_total", lambda x: round(x / 1000, 2),
        nome_saida="receita_em_milhares"
    )
    df_limpo = processar_coluna(
        df_limpo, "quantidade",
        lambda q: "Alto Volume" if q > 5 else "Baixo Volume",
        nome_saida="perfil_volume"
    )
    print(df_limpo[["receita_total", "receita_em_milhares", "quantidade", "perfil_volume"]].head().to_string(index=False))

    # Gerar gráficos PNG
    print("\n[7/8] Gerando gráficos...")
    gerar_graficos(df_limpo, metricas, pasta_saida=os.path.join("outputs", "graficos"))

    # Exportar tabelas e estatísticas
    print("\n[8/8] Exportando resultados...")
    estatisticas = {
        "total_registros": int(len(df_limpo)),
        "total_clientes": int(df_limpo["cliente"].nunique()),
        "receita_total": float(df_limpo["receita_total"].sum()),
        "quantidade_total_vendida": int(df_limpo["quantidade"].sum()),
        "preco_medio_unitario": float(df_limpo["preco_unitario"].mean()),
        "registros_excluidos_datas_invalidas": int(relatorio_limpeza["registros_excluidos"]),
        "clientes_bronze": int((clientes["segmento"] == "Bronze").sum()),
        "clientes_prata": int((clientes["segmento"] == "Prata").sum()),
        "clientes_ouro": int((clientes["segmento"] == "Ouro").sum())
    }
    exportar_resultados(metricas, clientes, estatisticas, pasta_saida="outputs")

    print("\n=== RESUMO FINAL ===")
    print(f"Receita total: R$ {estatisticas['receita_total']:,.2f}")
    print(f"Total de clientes: {estatisticas['total_clientes']}")
    print(f"Clientes Bronze: {estatisticas['clientes_bronze']}")
    print(f"Clientes Prata: {estatisticas['clientes_prata']}")
    print(f"Clientes Ouro: {estatisticas['clientes_ouro']}")
    print("\n[CONCLUÍDO] Fluxo finalizado com sucesso!")


# Ponto de entrada do projeto
if __name__ == "__main__":
    main()




 SALESINSIGHT PY - Análise de Dados de Vendas

[1/8] Carregando e inspecionando o dataset...
Registros originais: 2000
 id_venda    data_venda     cliente    produto    categoria   regiao  quantidade  preco_unitario
        1    2025-05-21 Cliente_016      Mouse  Perifericos  Sudeste           2          102.90
        2 DATA INVALIDA Cliente_003      Mouse  Perifericos      Sul           9          105.13
        3    2025-10-15 Cliente_013 Smartphone    Celulares    Norte           9         2267.33
        4    2025-05-23 Cliente_001    Monitor Computadores      Sul           4         1181.72
        5    2025-03-21 Cliente_014      Mouse  Perifericos Nordeste           7          114.25

Tipos de dados:
id_venda            int64
data_venda            str
cliente               str
produto               str
categoria             str
regiao                str
quantidade          int64
preco_unitario    float64
dtype: object

[2/8] Limpando e validando datas...
{'registros_iniciais': 